## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **4aa1b310-68c7-4ba7-be03-3fc80c5ace30**


## Dataset

# Swiss Journey ID &#40;SJYID&#41; 2026


## Description

&#42;&#42;WICHTIG: Dieser Datensatz wird per 14. Dezember 2026 eingestellt, da nach unserem Kenntnisstand kein Bedarf mehr dafür besteht. Bei Unklarheiten und Fragen wenden Sie sich bitte an opendata@sbb.ch.&#42;&#42;<br><br>Diese Dateien helfen dabei, die fachlichen &#40;VM-Nummer, TU-Code, Variante/Datum&#41; und die technischen Swiss Journey ID &#40;SJYID&#41; Identifikatoren in Beziehung zueinander zu setzen. Denn die SJYID ist nicht über all diese Werte hinweg eindeutig. Dadurch können Sie bei Bedarf die technischen Identifikatoren anhand der fachlichen Identifikatoren finden oder umgekehrt.<br><br>Diese neue Version 2 &#40;V2, seit 2025-11-25&#41; präsentiert die Daten neu als syntaktisch korrekte, reguläre CSV-Datei.<br><br>Daten für 2025 und älter sind in unserem Datenarchiv zu finden: https://archive.opentransportdata.swiss


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/swiss-journey-id-sjyid-2026-v2)


## Metadata

- **identifier:** list-sjyid-2026-v2@oevch
- **publisher.name:** 
- **organization.display&#95;name:** Geschäftsstelle Systemaufgaben Kundeninformation &#40;SBB AG&#41;
- **organization.url:** https://www.opentransportdata.swiss
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** 
- **issued:** 2025-12-04 23:00:00
- **metadata&#95;created:** 2025-12-06T01:47:43.042270
- **metadata&#95;modified:** 2026-11-15T08:08:15.778550
- **Resource id:** 645fde16-1e46-491c-b2be-cb10239502fb
- **Resource name:** Listing-SJYID-FP-2026-202603312207-v2.csv
- **Resource package&#95;id:** 4aa1b310-68c7-4ba7-be03-3fc80c5ace30
- **Resource description:** 
- **Resource issued:** 2026-04-07 05:00:18.835146
- **Resource modified:** 2026-04-07 05:00:18.835146
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;by
- **Resource rights:** https://opendata.swiss/terms-of-use&#35;terms&#95;by


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                # Detect file compression after HTTP compression has been decoded.
                if kwargs.get("compression", "infer") == "infer" and source.peek(2).startswith(
                    b"\x1f\x8b"
                ):
                    kwargs["compression"] = "gzip"
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **Listing-SJYID-FP-2026-202603312207-v2.csv** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://data.opentransportdata.swiss/dataset/69daee66-9c6c-4476-89c0-129888071a3a/resource/26305442-a86d-4d76-a998-55debf078f06/download/listing-sjyid-fp-2026-202603312207-v2.csv")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** opendata@sbb.ch &#124; opendata@sbb.ch
